In [4]:
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
load_dotenv()
import os

api_key = os.getenv("GOOGLE_API_KEY")

if os.environ['GOOGLE_API_KEY']:
    print("Google api key is set")
else:
    raise ValueError("Gemini API key is not set")


llm = ChatGoogleGenerativeAI(model="gemini-3.6-flash", google_api_key=api_key, temperature=0.7)

Google api key is set


In [5]:
from langchain_core.tools import tool

@tool
def resize_image(height: int, width: int, image_path: str) -> str:
    """
    Tool to resize image
    Args:
        height: Target height of the image
        width: Target width of the image
        image_path: Location of the target image
    """
@tool
def convert_image(image_path: str, format: str) -> str:
    """
    Tool to convert image type(jpg, jpeg, png, wpeg)
    Args:
        format: Target conversion type of image
        image_path: Location of the target image
    """
@tool
def compress_image(quality: int, image_path: str) -> str:
    """
    Tool to compress image
    Args:
        quality: after compression quality of image
        image_path: Location of the target image
    """
@tool
def rotate_image(rotation_angle: int, image_path: str) -> str:
    """
    Tool to rotate image
    Args:
        rotation_angle: Angle at which image is to be rotated
        image_path: Location of the target image
    """
@tool
def metadata_extraction_image(height: str, width: str, image_path: str) -> str:
    """
    Tool to extract metadata of image
    Args:
        height: Target height of the image
        width: Target width of the image
        image_path: Location of the target image
    """

In [ ]:
tools = [resize_image, convert_image, compress_image, rotate_image, metadata_extraction_image]

llm_with_tools = llm.bind_tools(tools)

In [ ]:
from typing import TypedDict, Literal, Optional, List, Annotated
from pydantic import BaseModel, Field
from langgraph.graph.message import add_messages

class ImageProcessingRequest(BaseModel):
    operation: Literal[
        "resize",
        "convert",
        "compress",
        "rotate",
        "metadata"
    ] = Field("Image processing operation to perform")

    image_path: str = Field(
        description="Path of the image file"
    )

    height: Optional[int] = Field(
        default=None,
        description="New image height for resize operation"
    )

    width: Optional[int] = Field(
        default=None,
        description="New image width for resize operation"
    )
    format: Optional[str] = Field(
        default=None,
        description="Target image format like JPG, PNG, WEBP"
    )

    quality: Optional[int] = Field(
        default=None,
        description="Compression quality from 1-100"
    )

    rotation_angle: Optional[int] = Field(
        default=None,
        description="Rotation angle like 90, 180, 270 degrees"
    )

class graph_schema(TypedDict):
    messages: Annotated[list, add_messages]
    image_path: str
    operation: str
    result: str

llm_with_schema = llm_with_tools.with_structured_output(ImageProcessingRequest)




ModuleNotFoundError: No module named 'lannggraph'

In [ ]:
def llm_node(state: graph_schema) -> graph_schema:
    message = state['messages']
    response = llm_with_schema.invoke(message)
    state['image_path'] = response.image_path
    state['operation'] = response.operation
    state['messages'] = [response]
    return state

def tool_node(state: graph_schema) -> graph_schema:
    message = state['messages']